# Package 설치

In [1]:
%pip install transformers tokenizers datasets accelerate sentencepiece pillow  timm -qU

Note: you may need to restart the kernel to use updated packages.


# Hugging Face Pipeline을 이용한 모델 활용

- Pipeline은 Transformers 라이브러리의 가장 기본적인 객체로, **전처리 - 추론 -> 후처리** 로 이어지는 일련의 과정을 자동화하여 손쉽게 모델을 사용할 수 있게 해준다.
- Task에 따라 다양한 Pipeline 클래스를 제공하며 `pipeline` 함수를 이용해 쉽게 생성할 수 있다.
- **task만 지정**해서 기본 제공 모델과 토크나이저를 사용하거나 **직접 모델과 토크나이저를 지정**해 생성할 수 있다.
- https://huggingface.co/docs/transformers/pipeline_tutorial

![huggingface_pipeline.png](figures/huggingface_pipeline.png)

## 지원하는 주요 태스크
- https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.pipeline.task
### 자연어 처리 태스크
- **text-classification**: 텍스트 분류
- **text-generation**: 텍스트 생성
- **translation**: 번역
- **summarization**: 요약
- **question-answering**: 질의응답
- **fill-mask**: 마스크 토큰 채우기
- **token-classification**: 개체명 인식, Pos tagging 같이 개별 토큰에 대한 분류
- **feature-extraction**: 특징 추출(context vector)

### 영상 처리 태스크
- **image-classification**: 이미지 분류
- **object-detection**
  -  객체 검출 (Object Detection)
  -  이미지 안에서 객체들의 위치와 class를 찾아내는 작업
- **image-segmentation**
  -  이미지 세분화 (Image Segmentation)
  -  이미지를 픽셀 단위로 분할하여 각 픽셀이 어떤 객체에 속하는지 분류하는 작업

## 모델 검색
![huggingface_model_search.png](figures/huggingface_model_search.png)



## pipeline 함수
- 주요파라미터
  - **task:** 수행하려는 작업의 유형을 문자열로 지정한다.
  - **model:**
    - 사용할 사전 학습된 모델의 이름 또는 경로를 지정한다. 
    - 모델이름(ID)은 `[모델소유자이름]/[모델이름]` 형식이다. Hugging Face에서 제공하는 모델의 경우는 `모델소유자이름`이 생략되어 있다. (ex: "google/gemma-2-2b", "gpt2")
    - 모델을 명시적으로 지정하지 않으면, **task에 맞는 기본 모델이 로드**된다.
  - **tokenizer:** 자연어 task에서 사용할 토크나이저를 지정한다. 생략하면 모델과 같이 제공되는(model과 이름이 같은 토크나이저) 토크나이저를 사용한다.
  - **framework:** 사용할 딥러닝 프레임워크를 지정한다. 'pt'는 PyTorch(Default), 'tf'는 TensorFlow를 지정한다.
  - **device:** Pipeline 모델을 실행할 디바이스를 지정한다. 문자열로 `"cpu", "cuda:1", "mps"`, 또는 GPU 번호를 정수로 지정한다. 
  - **revision:** 모델의 특정 버전을 지정할 때 사용한다.
  - **trust_remote_code:** hub 모델을 직접 다운 받는 것이 아니라 모델을 다운 받는 **코드**를 다운 받아 local에서 실행하는 경우 코드를 실행할 수있게 할 지 여부. (bool)
  - **use_fast:** 
    - 빠른 토크나이저를 사용할지 여부를 지정합니다. 기본값은 True입니다.
    - 빠른 토크나이저는 `Rust` 언어로 구현되어 속도가 빠르다. 단 모든 모델에 대해 지원하지 않는다. 지원하지 않을 경우 `use_fast=True`로 설정해도 일반 토크나이저가 사용된다.

## Task 별 pipeline 실습

### 텍스트 분류

In [1]:
from transformers import pipeline

In [3]:
# Model과 Tokenizer를 불러와 Pipeline을 생성 - task만 정해주면 나머지는 Default로 
pipe = pipeline(
    task='text-classification',
    framework='pt',
)
pipe

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


In [6]:
result = pipe("I am very happy")
result2 = pipe("I am happy")
print(result, result2)

[{'label': 'POSITIVE', 'score': 0.9998795986175537}] [{'label': 'POSITIVE', 'score': 0.9998801946640015}]


In [17]:
data = [ 
    "The project was completed successfully.", 
    "She always brings positive energy to the team.", 
    "I am confident that we will achieve our goals.",
    "The results were not as expected.", 
    "He struggled to meet the deadline.", 
    "The client was dissatisfied with the final product." 
]
result_list = pipe(data)
result_list

[{'label': 'POSITIVE', 'score': 0.9998227953910828},
 {'label': 'POSITIVE', 'score': 0.9998812675476074},
 {'label': 'POSITIVE', 'score': 0.9998470544815063},
 {'label': 'NEGATIVE', 'score': 0.9978100657463074},
 {'label': 'NEGATIVE', 'score': 0.99960857629776},
 {'label': 'NEGATIVE', 'score': 0.9996129870414734}]

In [104]:
# 특정 모델을 지정하여 사용 - hugging face에 등록된 모델 ID
# 모델 ID 형식 - (모델소유자ID)/모델ID
## (모델소유자ID) 가 생략된 경우 : hugging face 자체 모델 
model="distilbert-base-uncased-finetuned-sst-2-english"

pipe2 = pipeline(
    task='text-classification',
    framework='pt',
    model=model,                 # 사용할 모델을 지정. Hugging Face의 모델 ID, 로컬에 저장된 모델 저장 파일 경로.
    tokenizer=model              # 사용할 tokenizer를 지정. Hugging Face의 Tokenizer ID, 로컬에 저장된 tokenizer 파일 경로.
    # 대부분의 경우 둘이 동일한데 이떄는 생략 
)

result_list = pipe(data)
result_list

Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9998227953910828},
 {'label': 'POSITIVE', 'score': 0.9998812675476074},
 {'label': 'POSITIVE', 'score': 0.9998470544815063},
 {'label': 'NEGATIVE', 'score': 0.9978100657463074},
 {'label': 'NEGATIVE', 'score': 0.99960857629776},
 {'label': 'NEGATIVE', 'score': 0.9996129870414734}]

In [106]:
text = '민초'
result = pipe2(text)
result

[{'label': 'POSITIVE', 'score': 0.9262546896934509}]

In [22]:
kor_texts = [
    "이 영화 정말 재미있어요!",
    "서비스가 별로였어요.",
    "제품 품질이 우수합니다.",
    "따듯하고 부드럽고 제품은 너무 좋습니다. 그런데 배송이 너무 늦네요."  # 애매한 것 0.56 정도 나오네.
]
result_list = pipe(kor_texts)
result_list     # English 기반 학습을 한 모델이라 한국어에 대한 성능이 나오지 않아.

[{'label': 'POSITIVE', 'score': 0.9855567812919617},
 {'label': 'POSITIVE', 'score': 0.7425776124000549},
 {'label': 'POSITIVE', 'score': 0.6555715799331665},
 {'label': 'NEGATIVE', 'score': 0.5247918367385864}]

In [24]:
# 한국어로 학습된 모델
model = 'Copycats/koelectra-base-v3-generalized-sentiment-analysis' 
pipe3 = pipeline(
    task='text-classification',
    framework='pt',
    model=model,
)
result_list = pipe(kor_texts)
result_list

Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9855567812919617},
 {'label': 'POSITIVE', 'score': 0.7425776124000549},
 {'label': 'POSITIVE', 'score': 0.6555715799331665},
 {'label': 'NEGATIVE', 'score': 0.5247918367385864}]

In [27]:
# Hugging Face에서 한국어, text-classification으로 모델 찾기
from transformers import pipeline

pipe = pipeline("text-classification", model="tabularisai/multilingual-sentiment-analysis")
result = pipe("오늘 날씨가 너무 좋다")
result # 이 모델은 Very Positive 와 Positive를 나누기에 확률값 자체는 낮을 수도 

Device set to use cpu


[{'label': 'Very Positive', 'score': 0.501676082611084},
 {'label': 'Negative', 'score': 0.5245440602302551},
 {'label': 'Very Positive', 'score': 0.6271345615386963},
 {'label': 'Very Positive', 'score': 0.5461603999137878}]

### 제로샷 분류
- 제로샷(Zero-shot)은 각 개별 작업에 대한 특정 교육 없이 작업을 수행할 수 있는 task다.
- 입력 텍스트와 함께 클래스 레이블을 제공하면 분류 작업을 한다.
- 모델은  `task`에서 `Zero-Shot` 으로 시작하는 task를 선택하여 검색한다.

In [4]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-classification", model="tabularisai/multilingual-sentiment-analysis")
text = ['우지훈', '기원준', '박현아']
# labels = ['호', '불호']
result = pipe(text)
result

Device set to use cpu


[{'label': 'Very Positive', 'score': 0.4244690537452698},
 {'label': 'Neutral', 'score': 0.3499472141265869},
 {'label': 'Very Negative', 'score': 0.29877161979675293}]

In [30]:
%pip install hf_xet

   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.7 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.7 MB 1.1 MB/s eta 0:00:02
   ----------- ---------------------------- 0.8/2.7 MB 1.2 MB/s eta 0:00:02
   --------------- ------------------------ 1.0/2.7 MB 1.4 MB/s eta 0:00:02
   ------------------- -------------------- 1.3/2.7 MB 1.3 MB/s eta 0:00:02
   -------------------------- ------------- 1.8/2.7 MB 1.5 MB/s eta 0:00:01
   ------------------------------ --------- 2.1/2.7 MB 1.5 MB/s eta 0:00:01
   ---------------------------------------- 2.7/2.7 MB 1.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [5]:
from transformers import pipeline

model = "facebook/bart-large-mnli"

text = ["Python is a programming language.", 
        "I love soccer", 
        "The stock price rose slightly today."]
labels = ['IT', 'Sports']               # 이 Label중에 하나로 Shot ! 

pipe = pipeline(
        task='zero-shot-classification',
        model=model
)

result = pipe(text, labels)
result

Device set to use cpu


[{'sequence': 'Python is a programming language.',
  'labels': ['IT', 'Sports'],
  'scores': [0.5758535265922546, 0.4241464138031006]},
 {'sequence': 'I love soccer',
  'labels': ['Sports', 'IT'],
  'scores': [0.9935312867164612, 0.006468690931797028]},
 {'sequence': 'The stock price rose slightly today.',
  'labels': ['IT', 'Sports'],
  'scores': [0.6849520802497864, 0.3150479197502136]}]

In [34]:
# 다른 레이블 목록으로 해보자
labels2 = ['business', 'programming', 'sports', 'movie', 'education']
result = pipe(text, candidate_labels=labels2)
result

[{'sequence': 'Python is a programming language.',
  'labels': ['programming', 'business', 'movie', 'sports', 'education'],
  'scores': [0.9856367111206055,
   0.005072721280157566,
   0.0034023483749479055,
   0.002961924998089671,
   0.0029262355528771877]},
 {'sequence': 'I love soccer',
  'labels': ['sports', 'programming', 'business', 'movie', 'education'],
  'scores': [0.9952405691146851,
   0.0012840895215049386,
   0.0012676474871113896,
   0.0012649551499634981,
   0.0009427034528926015]},
 {'sequence': 'The stock price rose slightly today.',
  'labels': ['business', 'movie', 'programming', 'sports', 'education'],
  'scores': [0.7462778091430664,
   0.06974831968545914,
   0.06889291107654572,
   0.0645080953836441,
   0.05057287961244583]}]

### 텍스트 생성

In [36]:
pipe = pipeline(task='text-generation')

No model was supplied, defaulted to openai-community/gpt2 and revision 607a30d (https://huggingface.co/openai-community/gpt2).
Using a pipeline without specifying a model name and revision in production is not recommended.
c:\Users\jinhy\anaconda3\envs\dl\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jinhy\.cache\huggingface\hub\models--openai-community--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://d

In [37]:
start_text = "Today weather"
sent = pipe(start_text)
sent

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'Today weather in New York City is getting worse.\n\nA meteorological report released by the New York City Meteorological Department on Tuesday morning estimated that the weather pattern in New York City could be as bad as that in the entire nation. Though the temperature and humidity in some parts of the city will remain relatively constant, temperatures in the surrounding areas will be much lower as temperatures drop.\n\nThe weather forecast for the New York City area was based on three different weather patterns: 1) A high wind gust (15 mph or more) that is as high as 110 mph in the upper Midwest as it is in the Northeast, 2) a high wind gust (30 mph or more) that is as high as 80 mph in the Central Valley as it is in the West, and 3) a high wind gust (40 mph or more) at the same time as a rainstorm.\n\nA high wind gust is the most common wind gust in the region, and it is one of the major causes of thunderstorms. It is also the most likely for tropical storms, w

In [39]:
print(sent[0]['generated_text'])

Today weather in New York City is getting worse.

A meteorological report released by the New York City Meteorological Department on Tuesday morning estimated that the weather pattern in New York City could be as bad as that in the entire nation. Though the temperature and humidity in some parts of the city will remain relatively constant, temperatures in the surrounding areas will be much lower as temperatures drop.

The weather forecast for the New York City area was based on three different weather patterns: 1) A high wind gust (15 mph or more) that is as high as 110 mph in the upper Midwest as it is in the Northeast, 2) a high wind gust (30 mph or more) that is as high as 80 mph in the Central Valley as it is in the West, and 3) a high wind gust (40 mph or more) at the same time as a rainstorm.

A high wind gust is the most common wind gust in the region, and it is one of the major causes of thunderstorms. It is also the most likely for tropical storms, which can cause tornadoes, h

In [41]:
pipe(["I am", "Python is", "LLM is"])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[[{'generated_text': "I am a fan of many things - I've always been an audiophile, and I've been listening to some of the audiobooks myself. My main influences are the Beatles, Alice in Chains and The Beatles - but I've only ever listened to them once, and I've never been a fan of any of those. I'm a huge Beatles fan, and I've got to say I've always been an audiophile, so my first thing I've read about on the Internet is the Beatles. My first book was a Christmas card book - and that was so great, and I'm so glad I listened to it. I think I picked up a couple of them and read them all, and it was great. I could read a lot of books - but nothing that I've read. I've never read anything that has been written by an American artist or a person from the Beatles. I've always been an audiophile, though.\n\nAnd then there are the books by Harry Potter. I never read any of the books that were written by Harry Potter - and I've been reading a lot of those. I've read A Clockwork Orange, and it is 

In [46]:
# 이 모델은 한국어 학습 X
print(pipe('나는 어제')[0]['generated_text'])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


나는 어제어보에 어제어보에 파를 상터니다.

You have to understand that the person you have met is a person who you have met in the past.

In the past, you are constantly surrounded by people. You are constantly surrounded by people.

But now, you are surrounded by a person who you can't see.

A person is someone who you can't see.

Even now, you are surrounded by people who you can't see.

It's like you are standing in a pool.

But this is no time for swimming.

You are standing on the spot.

You are standing in front of the mirror.

And you are standing on the spot.

In the past, you were surrounded by people. But now, you are surrounded by people who you can't see.

You are surrounded by people who you can't see.

You are surrounded by people who you can't see.

You're surrounded by people who you can't see.

You're surrounded by people who you


In [107]:
from transformers import pipeline
model_id = 'Qwen/Qwen3-0.6B'
pipe = pipeline(task='text-generation', model=model_id)
pipe

Device set to use cpu


In [112]:
text = '우리가 먹는 민트초코 맛은 '
result = pipe(text, max_new_tokens=1000)
result

[{'generated_text': '우리가 먹는 민트초코 맛은 12개, 3개를 줄이는 경우, 12개가 13개보다 1000원 더 가격이 높은 경우, 12개가 13개보다 저렴한 경우, 어떤 것이 옳고, 어떤 것이 옳지 않은가?\n\nA. 12개가 13개보다 1000원 더 가격이 높은 경우\n\nB. 12개가 13개보다 1000원 저렴한 경우\n\nC. 12개가 13개보다 1000원 더 가격이 높은 경우\n\nD. 12개가 13개보다 1000원 저렴한 경우\n\nAnswer:\n\nA and C\n\nSo, the correct answer is A and C, which means the options are:\n\nA. 12개가 13개보다 1000원 더 가격이 높은 경우\n\nC. 12개가 13개보다 1000원 더 가격이 높은 경우\n\nSo, the answer is A and C.\n\n**Final Answer**\nA. 12개가 13개보다 1000원 더 가격이 높은 경우  \nC. 12개가 13개보다 1000원 더 가격이 높은 경우\n</s>\nThe correct answers are A and C.\n\nA. 12개가 13개보다 1000원 더 가격이 높은 경우  \nC. 12개가 13개보다 1000원 더 가격이 높은 경우\n\nThe correct answer is \\boxed{A} and \\boxed{C}. Therefore, the answer is \\boxed{A} and \\boxed{C}.\n</s>\nThe correct answers are A and C.\n\nA. 12개가 13개보다 1000원 더 가격이 높은 경우  \nC. 12개가 13개보다 1000원 더 가격이 높은 경우\n\nThe correct answer is \\boxed{A} and \\boxed{C}.\n\n</s>\n**Final Answer**\nA and C\n</s>\nThe correct answer is A and C.\n\nA. 12개가 13개보다 1

In [113]:
print(result[0]['generated_text'])

우리가 먹는 민트초코 맛은 12개, 3개를 줄이는 경우, 12개가 13개보다 1000원 더 가격이 높은 경우, 12개가 13개보다 저렴한 경우, 어떤 것이 옳고, 어떤 것이 옳지 않은가?

A. 12개가 13개보다 1000원 더 가격이 높은 경우

B. 12개가 13개보다 1000원 저렴한 경우

C. 12개가 13개보다 1000원 더 가격이 높은 경우

D. 12개가 13개보다 1000원 저렴한 경우

Answer:

A and C

So, the correct answer is A and C, which means the options are:

A. 12개가 13개보다 1000원 더 가격이 높은 경우

C. 12개가 13개보다 1000원 더 가격이 높은 경우

So, the answer is A and C.

**Final Answer**
A. 12개가 13개보다 1000원 더 가격이 높은 경우  
C. 12개가 13개보다 1000원 더 가격이 높은 경우
</s>
The correct answers are A and C.

A. 12개가 13개보다 1000원 더 가격이 높은 경우  
C. 12개가 13개보다 1000원 더 가격이 높은 경우

The correct answer is \boxed{A} and \boxed{C}. Therefore, the answer is \boxed{A} and \boxed{C}.
</s>
The correct answers are A and C.

A. 12개가 13개보다 1000원 더 가격이 높은 경우  
C. 12개가 13개보다 1000원 더 가격이 높은 경우

The correct answer is \boxed{A} and \boxed{C}.

</s>
**Final Answer**
A and C
</s>
The correct answer is A and C.

A. 12개가 13개보다 1000원 더 가격이 높은 경우  
C. 12개가 13개보다 1000원 더 가격이 높은 경우

The correct answer is

In [62]:
msa = [
    {'role':'user', 'content':'LLM에 대해서 설명해줘'}
]
result = pipe(msa, max_new_tokens=1000)
result

[{'generated_text': [{'role': 'user', 'content': 'LLM에 대해서 설명해줘'},
   {'role': 'assistant',
    'content': "<think>\nOkay, the user is asking for an explanation of LLMs. I need to start by defining what an LLM is. Let me recall that LLM stands for Large Language Model. They are big language models, right? So I should mention their purpose, like answering questions or generating text.\n\nNext, I should explain what makes them so big. The model is trained on massive amounts of text, so it can understand and process a lot of information. Also, they have a lot of knowledge and can learn from that. Maybe compare them to humans in terms of capability and speed.\n\nI should mention their applications. Common uses include writing, research, content creation, and even answering complex questions. It's important to highlight how they can handle various tasks. Oh, and maybe their limitations, like not being perfect in all areas or needing human input for some tasks. That shows the balance between

In [68]:
print(result[0]['generated_text'][1]['content'])

<think>
Okay, the user is asking for an explanation of LLMs. I need to start by defining what an LLM is. Let me recall that LLM stands for Large Language Model. They are big language models, right? So I should mention their purpose, like answering questions or generating text.

Next, I should explain what makes them so big. The model is trained on massive amounts of text, so it can understand and process a lot of information. Also, they have a lot of knowledge and can learn from that. Maybe compare them to humans in terms of capability and speed.

I should mention their applications. Common uses include writing, research, content creation, and even answering complex questions. It's important to highlight how they can handle various tasks. Oh, and maybe their limitations, like not being perfect in all areas or needing human input for some tasks. That shows the balance between their capabilities and the need for human assistance.

Wait, did I cover all the aspects? Let me check. Definiti

### 마스크 채우기

In [3]:
%pip install transformers

Note: you may need to restart the kernel to use updated packages.


In [4]:
from transformers import pipeline
text = "I'm going to <mask> because <mask> am hurt."
model = "distilroberta-base"

pipe = pipeline(
    task='fill-mask',
    model=model
)
result = pipe(text, top_k=2) # 확률 높은 단어 2개 찾기
type(result), result

Some weights of the model checkpoint at distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


(list,
 [[{'score': 0.26520195603370667,
    'token': 8930,
    'token_str': ' cry',
    'sequence': "<s>I'm going to cry because<mask> am hurt.</s>"},
   {'score': 0.06089078262448311,
    'token': 3581,
    'token_str': ' sleep',
    'sequence': "<s>I'm going to sleep because<mask> am hurt.</s>"}],
  [{'score': 0.9930052161216736,
    'token': 38,
    'token_str': ' I',
    'sequence': "<s>I'm going to<mask> because I am hurt.</s>"},
   {'score': 0.006336321122944355,
    'token': 939,
    'token_str': ' i',
    'sequence': "<s>I'm going to<mask> because i am hurt.</s>"}]])

In [72]:
text = "오늘 밤은 전국이 흐린 가운데 대부분 지역에 [MASK]가 내리겠고, 기온이 내려가면서 점차 [MASK]이 오는 곳이 많겠습니다"
model = 'beomi/kcbert-base'  # 영문 모델
pipe = pipeline(
    task='fill-mask',
    model=model
)
result = pipe(text)
result

c:\Users\jinhy\anaconda3\envs\dl\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jinhy\.cache\huggingface\hub\models--beomi--kcbert-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download

[[{'score': 0.6340426802635193,
   'token': 4072,
   'token_str': '##서',
   'sequence': '[CLS] 오늘 밤은 전국이 흐린 가운데 대부분 지역에서 가 내리겠고, 기온이 내려가면서 점차 [MASK] 이 오는 곳이 많겠습니다 [SEP]'},
  {'score': 0.11311697214841843,
   'token': 28206,
   'token_str': '비가',
   'sequence': '[CLS] 오늘 밤은 전국이 흐린 가운데 대부분 지역에 비가 가 내리겠고, 기온이 내려가면서 점차 [MASK] 이 오는 곳이 많겠습니다 [SEP]'},
  {'score': 0.037142351269721985,
   'token': 12,
   'token_str': ')',
   'sequence': '[CLS] 오늘 밤은 전국이 흐린 가운데 대부분 지역에 ) 가 내리겠고, 기온이 내려가면서 점차 [MASK] 이 오는 곳이 많겠습니다 [SEP]'},
  {'score': 0.03517227619886398,
   'token': 1664,
   'token_str': '비',
   'sequence': '[CLS] 오늘 밤은 전국이 흐린 가운데 대부분 지역에 비 가 내리겠고, 기온이 내려가면서 점차 [MASK] 이 오는 곳이 많겠습니다 [SEP]'},
  {'score': 0.019622188061475754,
   'token': 9666,
   'token_str': '##서는',
   'sequence': '[CLS] 오늘 밤은 전국이 흐린 가운데 대부분 지역에서는 가 내리겠고, 기온이 내려가면서 점차 [MASK] 이 오는 곳이 많겠습니다 [SEP]'}],
 [{'score': 0.10058404505252838,
   'token': 10108,
   'token_str': '바람',
   'sequence': '[CLS] 오늘 밤은 전국이 흐린 가운데 대부분 지역에 [MASK] 가 내리겠

### Token별 분류
- task: token-classification 
  - 개체명인식(ner), 품사부착(pos tagging)을 수행하는 task 
  - 개체명 인식은 문장에서 특정한 개체명(예: 사람 이름, 지명, 조직명 등)을 식별하는 task이다. 

In [73]:
text = "My name is Sylvain and I work at Hugging Face in Brooklyn."
model = "dbmdz/bert-large-cased-finetuned-conll03-english"
pipe = pipeline(task='token-classification', model=model)
result = pipe(text)
result

c:\Users\jinhy\anaconda3\envs\dl\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jinhy\.cache\huggingface\hub\models--dbmdz--bert-large-cased-finetuned-conll03-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Fallin

[{'entity': 'I-PER',
  'score': 0.99938285,
  'index': 4,
  'word': 'S',
  'start': 11,
  'end': 12},
 {'entity': 'I-PER',
  'score': 0.99815494,
  'index': 5,
  'word': '##yl',
  'start': 12,
  'end': 14},
 {'entity': 'I-PER',
  'score': 0.99590707,
  'index': 6,
  'word': '##va',
  'start': 14,
  'end': 16},
 {'entity': 'I-PER',
  'score': 0.99923277,
  'index': 7,
  'word': '##in',
  'start': 16,
  'end': 18},
 {'entity': 'I-ORG',
  'score': 0.9738931,
  'index': 12,
  'word': 'Hu',
  'start': 33,
  'end': 35},
 {'entity': 'I-ORG',
  'score': 0.976115,
  'index': 13,
  'word': '##gging',
  'start': 35,
  'end': 40},
 {'entity': 'I-ORG',
  'score': 0.9887976,
  'index': 14,
  'word': 'Face',
  'start': 41,
  'end': 45},
 {'entity': 'I-LOC',
  'score': 0.9932106,
  'index': 16,
  'word': 'Brooklyn',
  'start': 49,
  'end': 57}]

### 질의 응답
- 문서와 질문을 주면 문서에서 답을 찾아 응답한다.
- 이 경우 할루시네이션이 줄어듦? 없어짐? 쨋든

In [74]:
model = "distilbert-base-cased-distilled-squad"

question="Where do I work?"
# question="Where is Hugging Face?"
context="My name is Sylvain and I work at Hugging Face in Brooklyn"

In [75]:
pipe = pipeline(task='question-answering', model=model)

c:\Users\jinhy\anaconda3\envs\dl\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jinhy\.cache\huggingface\hub\models--distilbert-base-cased-distilled-squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to re

In [77]:
result = pipe(context=context, question=[question, 'Where is Hugging Face?'])
result

[{'score': 0.6949769854545593,
  'start': 33,
  'end': 45,
  'answer': 'Hugging Face'},
 {'score': 0.9893267154693604, 'start': 49, 'end': 57, 'answer': 'Brooklyn'}]

In [79]:
context = """우리나라 2대 수출 품목인 자동차가 도널드 트럼프 미국 행정부의 관세 여파로 지난달 큰 폭의 수출 감소율을 보이면서 우려가 커지고 있다. 현대차, 기아의 미국 수출 비중이 최대 85%에 이르는 상황에서 자동차 관세 장기화 시 피해는 걷잡을 수 없이 불어날 것이라는 암울한 전망이 나온다.
1일 산업통상자원부가 발표한 5월 수출입 동향에 따르면 지난달 자동차 수출은 작년 동기 대비 4.4% 감소한 62억달러로 집계됐다. 최대 자동차 시장인 미국으로의 수출은 18억4000만달러로 무려 32.0% 급감했다.
4월 미국의 수입산 자동차 25% 관세 부과에 이어 5월부터 일부 자동차 부품에도 25%의 관세가 적용된 결과다. 관세 장기화 시 피해는 더 커질 것이라는 우려가 현실화한 셈이다.
국내 완성차 1·2위 업체인 현대차·기아는 현지 생산 비중을 확대하는 동시에 가격 인상을 검토하고 있다. 관세 여파를 흡수하기 위해서다. 가격 인상이 현실화할 경우 미국 현지 판매는 줄어들 수밖에 없어 수출에는 더 악영향을 미칠 것으로 보인다.
"""

q1 = "현대차 기아의 미국 수출비중은?"
q2 = "자동차 수출이 얼마나 급감했나?"
q3 = "대미 수출 감소에 국내 자동차 업체들의 대응방법은?"
q4 = "가장 많이 사용된 특수문자는?"

In [82]:
model = "ainize/klue-bert-base-mrc"

pipe = pipeline(model=model, task='question-answering')
result = pipe(context=context, question=[q1, q2, q3, q4])
result

Device set to use cpu


[{'score': 0.6396173238754272, 'start': 99, 'end': 102, 'answer': '85%'},
 {'score': 0.632358193397522, 'start': 271, 'end': 276, 'answer': '32.0%'},
 {'score': 0.013693439774215221, 'start': 427, 'end': 433, 'answer': '가격 인상을'},
 {'score': 0.005158356856554747, 'start': 213, 'end': 217, 'answer': '4.4%'}]

### 문서 요약

In [87]:
model = "eenzeenee/t5-base-korean-summarization"
pipe = pipeline(model=model, task='summarization')

Device set to use cpu


In [88]:
result = pipe(context)

Token indices sequence length is longer than the specified maximum sequence length for this model (368 > 128). Running this sequence through the model will result in indexing errors


In [90]:
print(result[0]['summary_text'])

자동차가 트럼프 미국 행정부의 관세 여파로 큰 폭의 수출 감소율을 보이면서 자동차 관세 장기화 시 피해는 걷잡을 수 없이 불어날 것이라는 암울한 전망이 나온다.


### 번역

In [91]:
model = "Helsinki-NLP/opus-mt-fr-en"
text = "Ce cours est produit par Hugging Face."

In [92]:
model = "Helsinki-NLP/opus-mt-ko-en"
text = '롤 내전할 사람 구해요'
pipe = pipeline(model=model, task='translation')
result = pipe(text)
result

c:\Users\jinhy\anaconda3\envs\dl\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jinhy\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-ko-en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\jinhy\anaconda3\envs\dl\Lib\site-packages\transformers\models\marian\tokenization_marian.py:177: Use

[{'translation_text': 'I need someone to do the roll.'}]

### 이미지를 설명하는 텍스트 생성

In [95]:
url1 = "https://huggingface.co/datasets/Narsil/image_dummy/resolve/main/parrots.png"
url2 = "https://th.bing.com/th?id=ORMS.c526884bbea37c0bb9501f4f83b601e4&pid=Wdp&w=268&h=140&qlt=90&c=1&rs=1&dpr=1&p=0"
url3 = "http://images.cocodataset.org/val2017/000000039769.jpg"

In [96]:
model = "ydshieh/vit-gpt2-coco-en"
pipe = pipeline(model=model, task='image-to-text')

Device set to use cpu


In [98]:
result = pipe([url1, url2, url3])
result

[[{'generated_text': 'two birds are standing next to each other '}],
 [{'generated_text': 'a baseball player is throwing a ball '}],
 [{'generated_text': 'a cat laying on a blanket next to a cat laying on a bed '}]]

### 이미지 분류

In [ ]:
url = "https://pds.joongang.co.kr/news/component/htmlphoto_mmdata/202306/25/488f9638-800c-4bac-ad65-82877fbff79b.jpg"

In [99]:
model = "google/vit-base-patch16-224"
pipe = pipeline(model=model, task='image-classification')
result = pipe(url)
result

c:\Users\jinhy\anaconda3\envs\dl\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jinhy\.cache\huggingface\hub\models--google--vit-base-patch16-224. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTT

[{'label': 'Egyptian cat', 'score': 0.937441885471344},
 {'label': 'tabby, tabby cat', 'score': 0.038442447781562805},
 {'label': 'tiger cat', 'score': 0.014411325566470623},
 {'label': 'lynx, catamount', 'score': 0.0032743127085268497},
 {'label': 'Siamese cat, Siamese', 'score': 0.0006795904482714832}]

### Object Detection

In [6]:
image_path1 = r"data/image1.jpg"
image_path2 = r"data/image2.jpg"
image_path3 = r"data/image3.jpg"

model='facebook/detr-resnet-50'
pipe = pipeline(task='object-detection', model=model)
result = pipe([image_path1, image_path2, image_path3])
result

c:\Users\jinhy\anaconda3\envs\dl\Lib\site-packages\torch\nn\modules\module.py:2409: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
c:\Users\jinhy\anaconda3\envs\dl\Lib\site-packages\torch\nn\modules\module.py:2409: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
c:\Users\jinhy\anaconda3\envs\dl\Lib\site-packages\torch\nn\modules\module.py:2409: UserWarning: for bn1.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which

[[{'score': 0.9988904595375061,
   'label': 'dog',
   'box': {'xmin': 430, 'ymin': 423, 'xmax': 533, 'ymax': 597}},
  {'score': 0.9998466968536377,
   'label': 'person',
   'box': {'xmin': 531, 'ymin': 158, 'xmax': 673, 'ymax': 581}}],
 [{'score': 0.9981694221496582,
   'label': 'cat',
   'box': {'xmin': 541, 'ymin': 122, 'xmax': 719, 'ymax': 535}},
  {'score': 0.9980827569961548,
   'label': 'cat',
   'box': {'xmin': 198, 'ymin': 48, 'xmax': 373, 'ymax': 459}},
  {'score': 0.9971736669540405,
   'label': 'cat',
   'box': {'xmin': 0, 'ymin': 89, 'xmax': 255, 'ymax': 535}},
  {'score': 0.818007230758667,
   'label': 'bench',
   'box': {'xmin': 218, 'ymin': 355, 'xmax': 718, 'ymax': 535}},
  {'score': 0.9972655773162842,
   'label': 'cat',
   'box': {'xmin': 366, 'ymin': 60, 'xmax': 580, 'ymax': 479}}],
 [{'score': 0.9956639409065247,
   'label': 'cell phone',
   'box': {'xmin': 96, 'ymin': 165, 'xmax': 136, 'ymax': 236}},
  {'score': 0.9919518232345581,
   'label': 'tv',
   'box': {'xmi

In [7]:
# 분류 Class를 확인
pipe.model.config.id2label

{0: 'N/A',
 1: 'person',
 10: 'traffic light',
 11: 'fire hydrant',
 12: 'street sign',
 13: 'stop sign',
 14: 'parking meter',
 15: 'bench',
 16: 'bird',
 17: 'cat',
 18: 'dog',
 19: 'horse',
 2: 'bicycle',
 20: 'sheep',
 21: 'cow',
 22: 'elephant',
 23: 'bear',
 24: 'zebra',
 25: 'giraffe',
 26: 'hat',
 27: 'backpack',
 28: 'umbrella',
 29: 'shoe',
 3: 'car',
 30: 'eye glasses',
 31: 'handbag',
 32: 'tie',
 33: 'suitcase',
 34: 'frisbee',
 35: 'skis',
 36: 'snowboard',
 37: 'sports ball',
 38: 'kite',
 39: 'baseball bat',
 4: 'motorcycle',
 40: 'baseball glove',
 41: 'skateboard',
 42: 'surfboard',
 43: 'tennis racket',
 44: 'bottle',
 45: 'plate',
 46: 'wine glass',
 47: 'cup',
 48: 'fork',
 49: 'knife',
 5: 'airplane',
 50: 'spoon',
 51: 'bowl',
 52: 'banana',
 53: 'apple',
 54: 'sandwich',
 55: 'orange',
 56: 'broccoli',
 57: 'carrot',
 58: 'hot dog',
 59: 'pizza',
 6: 'bus',
 60: 'donut',
 61: 'cake',
 62: 'chair',
 63: 'couch',
 64: 'potted plant',
 65: 'bed',
 66: 'mirror',
 67

In [8]:
model = pipe.model
tokenizer = pipe.tokenizer

In [9]:
model

DetrForObjectDetection(
  (model): DetrModel(
    (backbone): DetrConvModel(
      (conv_encoder): DetrConvEncoder(
        (model): FeatureListNet(
          (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
          (bn1): DetrFrozenBatchNorm2d()
          (act1): ReLU(inplace=True)
          (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
          (layer1): Sequential(
            (0): Bottleneck(
              (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
              (bn1): DetrFrozenBatchNorm2d()
              (act1): ReLU(inplace=True)
              (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
              (bn2): DetrFrozenBatchNorm2d()
              (drop_block): Identity()
              (act2): ReLU(inplace=True)
              (aa): Identity()
              (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      

In [10]:
tokenizer